In [ ]:
!pip -q install yellowbrick

In [ ]:
import math
import random
import pickle
import operator
import itertools
import functools
import numpy as np
import pandas as pd
import seaborn as sns
import plotly.express as px
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GridSearchCV
from yellowbrick.classifier import ConfusionMatrix
from sklearn.model_selection import train_test_split
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Tratamento de Dados

Para poder utilizar alguma das portas, basta descomentar a linha referente a porta e colocar o número de entradas, tenha em mente que o número de entradas é $2^x$, ou seja, pode ser que demore muito para conseguir plotar o gráfico ou treinar os modelos

In [ ]:

#Número de entradas da tabela verdade
numEntradas = 4

entradas = list(itertools.product([0, 1], repeat=numEntradas))

#porta XOR
#saidas = [int(functools.reduce(operator.xor, entrada)) for entrada in entradas]

#porta AND
saidas = [int(all(entrada)) for entrada in entradas]

#porta OR
#saidas = [int(any(entrada)) for entrada in entradas]

df = pd.DataFrame(entradas, columns=[f'Entrada_{i}' for i in range(0, numEntradas)])
df['Saída'] = saidas

In [ ]:
tsne_results = TSNE(n_components=2, perplexity= 3, n_iter=3000).fit_transform(df)
#50 if (2**numEntradas - 2 > 50) else (2**numEntradas - 2)
result = pd.DataFrame(tsne_results, columns=['x', 'y'])


result['Saída'] = saidas
sns.scatterplot(data=result, x="x", y='y', hue='Saída')

plt.xlabel("eixo x")
plt.ylabel("eixo y")
plt.title('Visualização dos resultados do t-SNE')
plt.show()

### Tentativa com  Iris com um único neurônio

Caso for usar a Iris, rodar o bloco abaixo

In [ ]:
numEntradas = 4
iris_data = pd.read_csv("/content/sample_data/Iris.csv")
features = iris_data[["sepallength","sepalwidth","petallength"]]
#sepallength	sepalwidth	petallength	petalwidth	class

tsne_results = TSNE(n_components=2, perplexity= 5, n_iter=3000).fit_transform(features)
result_iris = pd.DataFrame(tsne_results, columns=['x', 'y'])
result_iris['Saída'] = iris_data["class"]
sns.scatterplot(data=result_iris, x="x", y='y', hue='Saída')

base = iris_data

plt.xlabel("eixo x")
plt.ylabel("eixo y")
plt.title('Visualização dos resultados do t-SNE')
plt.show()

### Separando atributos de entrada e de classe

In [ ]:
#ignorar se tiver usando iris como teste
base = df

In [ ]:
X_prev = base.iloc[:,0:numEntradas].values

In [ ]:
X_prev

In [ ]:
y_classe = base.iloc[:, numEntradas].values

In [ ]:
y_classe

In [ ]:
#Ignorar se não tiver usando iris
#le = LabelEncoder()
#y_classe = le.fit_transform(y_classe)

In [ ]:
y_classe

Existem dois tipos de definições de treino e testes:


*   Caso for uma porta lógica recomendo não usar a divisão pra treinos, pois o perceptron precisa de todos os casos para funcionar
*   Caso for outro tipo de dataset, talvez dividindo uma parte para treino e teste seja a melhor opção



In [ ]:
#portas lógicas
X_treino =  X_teste = X_prev
y_treino = y_teste = y_classe

#divisão para treino e teste
#X_treino, X_teste, y_treino, y_teste = train_test_split(X_prev, y_classe, test_size = 0.40, random_state=0, shuffle=True)

In [ ]:
X_treino

In [ ]:
y_treino

In [ ]:
X_teste

In [ ]:
y_teste

In [ ]:
with open('port.pkl', mode = 'wb') as f:
  pickle.dump([X_treino, X_teste, y_treino, y_teste], f)

# Perceptron


In [ ]:

class Perceptron(BaseEstimator, ClassifierMixin):
  def __init__(self,bias = 1,taxaAprendizagem= 0.3,epocas= 0,tipoAtivacao = "Binary Step",alpha = 1):
    self.bias =bias
    self.taxaAprendizagem = taxaAprendizagem
    self.epocas = epocas
    self.tipoAtivacao = tipoAtivacao
    self.alpha = alpha
    self.w = []


  def getPesos(self):
    return self.w

  def fit(self, X_treino, y_treino):
    self.trainPerceptron(X_treino,y_treino)

  def predict(self,X_teste):
    return self.testPerceptron(X_teste)

  def functionAtivacao(self,x):
    result = 0
    #print(x,self.w)
    for i in range(len(x)):
      result += x[i] * self.w[i]
    match self.tipoAtivacao:
      case "Binary Step":
        result  = 1 if ((result) > 0) else 0
      case "Sigmoidal":
        result = 1/(1+math.e**(-self.alpha*result))
      case _:
        result  = 1 if ((result) > 0) else 0
    return result

  def functionAjusteWeight(self,x,erro):
    for i in range(len(self.w)):
      #print("peso ",(self.w[i] + self.taxaAprendizagem * erro * x[i]), ' = ', self.w[i], ' + ' ,self.taxaAprendizagem, ' * ', erro ,' * ' ,x[i])
      self.w[i] = self.w[i] + self.taxaAprendizagem * erro * x[i]

  def randomWeight(self,tam):
    for i in range(0,tam):
      self.w.append(0.301)

  def trainPerceptron(self,X_treino,y_treino):
    bias_column = np.full((X_treino.shape[0], 1), self.bias)
    X_treino = np.hstack((bias_column, X_treino))
    self.randomWeight(len(X_treino[0]))
    erro_global = 0.001
    numEpocas = 0
    while(numEpocas < self.epocas and erro_global > 0 ):
      erro_global = 0
      for i in range(len(X_treino)):
        #print("funcao ativacao")
        saidaNeuronio = self.functionAtivacao(X_treino[i])
        erro = y_treino[i] - saidaNeuronio
        #print("erro = ",erro, ' = ',self.y_treino[i], ' - ',saidaNeuronio)
        erro_global +=abs(erro)
        if erro != 0:
         # print("ajustando peso:")
          self.functionAjusteWeight(X_treino[i],erro)
      #print("epoca: ",numEpocas)
      #print("erro_global =",erro_global)
      numEpocas+=1

  def testPerceptron(self,X_teste):
    result = []
    bias_column = np.full((X_teste.shape[0], 1), self.bias)
    X_teste =  np.hstack((bias_column, X_teste))
    for i in range(len(X_teste)):
      x = self.functionAtivacao(X_teste[i])
      result.append(x)
    return result


# Backpropagation

In [ ]:
class Backpropagation(BaseEstimator, ClassifierMixin):
  def __init__(self,bias = 1,taxaAprendizagem= 0.3,epocasBackpropagation= 1,
               tipoAtivacao = "Binary Step",alpha = 1,numNeuronios = 1,hiddensLayers = 1):
    self.X_treino = None
    self.y_treino = None
    self.bias =bias
    self.taxaAprendizagem = taxaAprendizagem
    self.epocasBackpropagation= epocasBackpropagation
    self.tipoAtivacao = tipoAtivacao
    self.alpha = alpha
    self.numNeuronios = numNeuronios
    self.hiddensLayers = hiddensLayers
    self.w = []
    self.camadas = []

  def initRede(self,tamX,numClasses): #tamX = 4 : 3num+1bias
    camada = []
    entradas = tamX
    for _ in range(self.hiddensLayers): #layers = 2 
      for _ in range(self.numNeuronios):  # num = 2
        p =Perceptron(bias = self.bias,taxaAprendizagem = self.taxaAprendizagem,
                              tipoAtivacao = self.tipoAtivacao,alpha= self.alpha)
        p.randomWeight(entradas)
        camada.append(p)
        
      entradas = self.numNeuronios + 1 # 1 = bias
      self.camadas.append(camada)
      camada = []
    for _ in range(numClasses):
      p =Perceptron(bias = self.bias,taxaAprendizagem = self.taxaAprendizagem,
                              tipoAtivacao = self.tipoAtivacao,alpha= self.alpha)
      p.randomWeight(entradas)
      camada.append(p)
    self.camadas.append(camada)
        

  def derivateSigmoide(self,x):
     return x * (1-x)

  def calculoErroOutput(self, erro,cAtual,ativacao):
    result = []
    for k in range(len(cAtual)):
      for j in range(self.numNeuronios):
        result.append(self.derivateSigmoide(ativacao[k])*(cAtual[k].w[j] * erro ))
      return result
  
  def calculoErro(self, cAtual,cAnterior, erro,ativacao):
    result = []
    soma = 0
    for k in range(len(cAnterior)):
      for i in range(len(cAnterior[k].w )- 1):
        soma+=cAnterior[i].w[k] * erro[k] 
      result.append(self.derivateSigmoide(ativacao[k])*soma)
      return result
      
      
      
  def functionAjusteWeight(self,c,erro,ativacao):
    for i in range(len(c)):
      c[i].functionAjusteWeight(ativacao,erro[i])

  def trainBackpropagation(self,X_treino,y_treino):
    nc = 1 if len(np.unique(y_treino)) == 2 else len(np.unique(y_treino)) # isso porque caso seja binario só é preciso um unico neuronio
    self.initRede((len(X_treino[0])+1),numClasses = nc)
    entradas = []
    ativacao = []
    s = []
    numEpocas = 0
    erro_global = 0.001
    i =0
    while(numEpocas<self.epocasBackpropagation and erro_global > 0):
      erro_global = 0
      for x in range(len(X_treino)):
        entradas = X_treino[x].tolist()
        for c in self.camadas:
          entradas.append(self.bias)
          ativacao.append(entradas)
          for ci in c:
              s.append(ci.functionAtivacao(entradas))
          entradas = s
          s = [] 
        ativacao.append(entradas)
        s = entradas


        #agora é o calculo de erro e ajuste de pesos
        erroParcial = y_treino[x] - s[0]
        if erroParcial != 0:
          erroOutput = []
          erroOutput.append(self.derivateSigmoide(ativacao[-1][0]) * erroParcial)
          self.functionAjusteWeight(self.camadas[-1],erroOutput,ativacao[len(ativacao)-2])
          erro = self.calculoErroOutput(erro =erroParcial,cAtual = self.camadas[-1],ativacao =ativacao[len(ativacao)-1])
          for i in range(len(self.camadas) - 2, 0, -1):
            self.functionAjusteWeight(c =self.camadas[i],erro = erro,ativacao = ativacao[i+1])
            erro = self.calculoErro(erro = erro,cAtual=self.camadas[i],cAnterior=self.camadas[i],ativacao=ativacao[i-1])
            
            i-=1
        erro_global +=abs(erroParcial)
















## Uso do Backpropagation

In [ ]:
with open('port.pkl', 'rb') as f:
  X_treinoB, X_testeB, y_treinoB, y_testeB = pickle.load(f)

In [ ]:
backPropagation = Backpropagation(bias =1,taxaAprendizagem=0.3 , epocasBackpropagation= 100, tipoAtivacao= "Sigmoidal",alpha=1,numNeuronios=2,hiddensLayers=2)

backPropagation.trainBackpropagation(X_treinoB,y_treinoB)


IndexError: list index out of range